# 09 · Advanced orchestration

**Course:** Building LLM Agents, notebook 9 of 10
**Where we are:** in notebook 08 we wired several agents together with fixed control flow: a sequential pipeline, a router that picks one worker, an orchestrator that fans work out. You decided the shape of those systems in advance. This notebook is about the opposite: letting the system decide its own next move at runtime.

### What you'll learn
- Planner-executor: have one agent write an explicit, inspectable plan, then have another agent carry it out step by step
- Reflection / self-critique: the single highest-value technique, *generate, critique, revise* in a loop until a quality gate passes
- Supervisor / manager: an agent that repeatedly decides which specialist to call next, based on the current state, until the goal is met
- The dangers of dynamic control flow (non-termination, cost blow-ups, error propagation) and the simple guards that keep them in check

### Prerequisites
- Notebooks 01 to 08. You should be comfortable with the agent loop and the `Agent` / `Tool` classes, and with notebook 08's four static patterns.

> Mental model: in notebook 08 you drew the flowchart and the agents filled in the boxes. Here the model draws the flowchart as it goes. That extra freedom is what makes these systems good, and it is also what lets them spin forever or burn tokens if you don't fence them in.

> Recap of notebook 08: you can already chain agents (pipeline), pick one of several with a router, split a job across workers under an orchestrator, and run agents in parallel then merge. In all of those, the control flow was static: written once, the same every run. Today we make the control flow dynamic, produced by the model at runtime.

Start from the usual setup cell.

In [ ]:
from helpers import client, MODEL, ask, jprint, show_messages, Agent, Tool
import json

print("model:", MODEL)

We'll lean on JSON mode a lot in this notebook. Every dynamic decision (a plan, a critic's verdict, a supervisor's next action) is something our code has to read and branch on, so we want clean, parseable JSON, not prose. Our endpoint supports `response_format={"type": "json_object"}`, so let's make a tiny helper for "ask the model and get back a dict".

In [ ]:
def ask_json(system, user, max_tokens=8000, temperature=0):
    """One LLM call that returns a parsed dict (JSON mode).

    We use temperature=0 because every call here is a *decision*, not
    creative writing — we want it predictable.
    """
    resp = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": system},
            {"role": "user", "content": user},
        ],
        temperature=temperature,
        max_tokens=max_tokens,
        response_format={"type": "json_object"},
    )
    return json.loads(resp.choices[0].message.content)

# quick check
ask_json("Reply with JSON only.", 'Return {"ok": true} and nothing else.')

## 1. Planner to executor

The first dynamic pattern splits a task into two phases handled by two roles:

1. **Planner**: reads the task and produces an explicit, ordered plan (a JSON list of steps). It does not solve anything; it only decides what to do and in what order.
2. **Executor**: walks the plan one step at a time, carrying out each step and passing its result forward to the next. The executor can be a plain LLM call, or, as in notebook 04/06, a full tool-using `Agent`.

> Mental model: think of the planner as the architect and the executor as the builder. The architect's drawing is visible. You can read it, check it, even edit it before a single brick is laid.

Let's write the planner first. The key design choice is forcing it into a strict JSON shape so our code can iterate over the steps.

![Planner emits a plan as data; executor runs it step by step](figures/09-planner-executor.svg)

*Figure: planning produces an inspectable plan; execution consumes it step by step. Generate it from the prompt in [`figures/FIGURES.md`](figures/FIGURES.md).*

In [ ]:
PLANNER_SYS = (
    "You are a planner. Break the user's task into 2-4 small, ordered, "
    "self-contained steps. Return JSON of the exact form "
    '{"steps": ["step 1", "step 2", ...]}. '
    "Each step is one short instruction. Do NOT solve the task; only plan it."
)

def plan_task(task):
    out = ask_json(PLANNER_SYS, f"Task: {task}", max_tokens=8000)
    return out["steps"]

task = "Write a two-line thank-you note to a volunteer named Sam who organised a beach cleanup."
steps = plan_task(task)
for i, s in enumerate(steps, 1):
    print(f"{i}. {s}")

That plan is a plain Python list, completely inspectable. Now the executor. The crucial detail is the carry-forward: each step is run with the overall task and the work done so far in its prompt, so step 2 can build on step 1's output.

In [ ]:
EXEC_SYS = (
    "You are an executor. You are given the overall task, the work done so far, "
    "and the current step. Carry out ONLY the current step and return its result. "
    "Be concise."
)

def execute_plan(task, steps, verbose=True):
    work_so_far = ""
    for i, step in enumerate(steps, 1):
        user = (
            f"Overall task: {task}\n\n"
            f"Work so far:\n{work_so_far or '(nothing yet)'}\n\n"
            f"Current step ({i}/{len(steps)}): {step}"
        )
        result = ask(user, system=EXEC_SYS, temperature=0, max_tokens=8000).strip()
        work_so_far += f"\n[step {i}] {result}"
        if verbose:
            print(f"-- step {i} -> {result}\n")
    return work_so_far.strip()

final = execute_plan(task, steps)
print("ASSEMBLED RESULT:\n" + final)

> Why separate planning from execution? Three concrete payoffs:
> - Clearer reasoning. The model commits to a strategy before it starts producing, instead of improvising. The plan is a cheap, short artefact to get right.
> - Inspectability. The plan is data you can read, log, validate, or show a human for approval before anything runs.
> - Replanning. If a step fails or the world changes, you can throw away the rest of the plan and ask the planner for a new one, without losing the work already done.

> Rule of thumb: use planner-executor when a task has several dependent steps and you'd like to see the strategy before committing. For a one-shot task it's overkill, and a single agent is fine.

> Gotcha: the plan is only a suggestion from the model. Validate it: check it's a non-empty list, cap its length, and be ready for a step that doesn't quite make sense. Never `exec()` a planner's output or trust it blindly.

![Reflection loop with a quality gate and a round cap](figures/09-reflection-gate.svg)

*Figure: generate, critique, gate (good?), return or revise, bounded by max_rounds. Generate it from the prompt in [`figures/FIGURES.md`](figures/FIGURES.md).*

## 2. Reflection / self-critique

If you take one technique from this notebook, take this one. Most of the quality you see in good agent systems comes from a simple idea: don't ship the first draft. Instead:

```
generate  ->  critique against explicit criteria  ->  revise
```

…and loop until a critic says the work is good, or you hit a round cap. The critic returns both a machine-readable verdict (so our loop can stop) and feedback (so the next draft can improve).

> Why you care: a model is often a better judge of its own output than a producer of it in one pass. Asking "is this good, and why not?" surfaces concrete flaws that a rewrite can then fix, often a big jump in quality for a couple of extra cheap calls.

First, the critic. The hard part is explicit criteria: vague critics drift, while strict ones with a checklist are reliable.

In [ ]:
CRITIC_SYS = (
    "You are a strict critic. Judge the DRAFT against the CRITERIA. "
    'Return JSON {"good": true|false, "feedback": "..."}. '
    "Set good=true only if EVERY criterion is clearly met. "
    "Keep feedback to one or two concrete, actionable sentences."
)

def critique(draft, criteria):
    return ask_json(
        CRITIC_SYS,
        f"CRITERIA:\n{criteria}\n\nDRAFT:\n{draft}",
        max_tokens=8000,
    )

# a deliberately weak draft, to see the critic bite
critique("Our bottle is a game-changer that will revolutionise how you stay hydrated all day long.",
         "1. Under 10 words. 2. Mentions a concrete benefit. 3. No cliches like 'game-changer'.")

The critic flagged real, specific problems. Now we wrap generate, critique, revise into a `reflect(...)` loop. Note the two ways it can stop: the quality gate (`good == True`) or the round cap. The cap is not optional.

In [ ]:
def reflect(task, criteria, max_rounds=3, verbose=True):
    """Generate, then critique-and-revise until the gate passes or rounds run out."""
    draft = ask(task, system="You are a helpful writer. Be concise.",
                temperature=0.4, max_tokens=8000).strip()
    if verbose:
        print(f"draft     : {draft}")

    for r in range(1, max_rounds + 1):
        verdict = critique(draft, criteria)
        if verbose:
            print(f"round {r}  : good={verdict['good']} | {verdict['feedback']}")
        if verdict.get("good"):
            return {"final": draft, "rounds": r, "passed": True}
        # revise using the critic's feedback
        draft = ask(
            f"Task: {task}\n\nYour previous draft:\n{draft}\n\n"
            f"A critic said: {verdict['feedback']}\n\n"
            "Rewrite the draft to fix that. Return only the improved version.",
            system="You are a helpful writer. Be concise.",
            temperature=0.4, max_tokens=8000,
        ).strip()
        if verbose:
            print(f"revision  : {draft}")

    return {"final": draft, "rounds": max_rounds, "passed": False}

Let's run it on a task with a clear, checkable rubric so the improvement is measurable. We can literally watch the gate flip from `False` to `True` and the draft get tighter.

In [ ]:
result = reflect(
    task="Write a single-sentence product tagline for a reusable water bottle.",
    criteria=("1. Under 10 words. "
              "2. Mentions a concrete benefit (not vague). "
              "3. No cliches like 'game-changer' or 'revolutionary'."),
    max_rounds=3,
)
print("\nFINAL :", result["final"])
print("rounds:", result["rounds"], "| gate passed:", result["passed"])

> Key idea: the loop has a producer (warmer temperature, allowed to be creative) and a judge (temperature 0, strict, machine-readable verdict). Separating those two jobs is what makes reflection work: the judge isn't attached to the words it's critiquing.

> Always cap the rounds. Two failure modes if you don't:
> - Infinite loop: the critic is never satisfied, the producer never quite there.
> - Condorcet / ping-pong: round 3 reintroduces the flaw round 1 fixed, round 4 fixes it again, forever.
>
> The `max_rounds` guard turns "loop until perfect" (unbounded) into "loop until perfect or N tries" (bounded). When you exhaust the rounds, return the best draft and the fact that the gate didn't pass, so the caller can decide what to do.

> Cost: each round is two extra LLM calls (critique and revise). Reflection is cheap per task but multiplies fast across many tasks, so reach for it where quality matters, not everywhere.

## 3. Supervisor / manager

Notebook 08's router made one decision: read the request, pick a worker, done. A supervisor makes that decision repeatedly. It loops: inspect the current state, decide which specialist to call next, run it, inspect again, and so on until it judges the goal met.

> Mental model: this is the agent loop from notebook 04 (reason, act, observe), except the "tools" the agent reaches for are whole other agents. The supervisor is a manager delegating to a team and watching the work pile up.

We'll coordinate two specialists from earlier in the course: a researcher and a writer. Each is just an `Agent` (here capped at a single step, since they don't need their own tools).

![A supervisor dynamically routes among agents in a loop](figures/09-supervisor.svg)

*Figure: the agent loop applied to choosing other agents, with a clean exit and a hard cap. Generate it from the prompt in [`figures/FIGURES.md`](figures/FIGURES.md).*

In [ ]:
researcher = Agent(
    name="researcher",
    system="You list 2-3 short factual bullet points about the topic. No preamble.",
    max_steps=1, temperature=0,
)
writer = Agent(
    name="writer",
    system="You turn the given notes into one tight paragraph (max 3 sentences).",
    max_steps=1, temperature=0,
)

# A registry the supervisor can dispatch to by name.
specialists = {
    "researcher": lambda x: researcher.run(x),
    "writer":     lambda x: writer.run(x),
}

Now the supervisor itself. On each turn it sees the goal and the state so far (what each specialist has produced), and returns a single next action: call a specialist, or declare `done`. Forcing it to choose one action per turn keeps the loop legible.

In [ ]:
SUP_SYS = (
    "You are a supervisor coordinating specialists to reach a GOAL. "
    "Available specialists: 'researcher' (gathers facts about a topic), "
    "'writer' (turns notes into a paragraph). "
    "Look at the STATE and decide the single next action. Return JSON of the form "
    '{"action": "researcher"|"writer"|"done", "input": "...", "reason": "..."}. '
    "Use 'done' only when the goal is fully met; put the final answer in 'input'."
)

def supervise(goal, max_steps=5, verbose=True):
    state = []  # list of (specialist_name, output)
    for step in range(1, max_steps + 1):
        state_str = "\n".join(f"- {who}: {out}" for who, out in state) or "(empty)"
        decision = ask_json(SUP_SYS, f"GOAL: {goal}\n\nSTATE so far:\n{state_str}", max_tokens=8000)
        action = decision["action"]
        if verbose:
            print(f"step {step}: action={action} | {decision.get('reason','')[:90]}")

        if action == "done":                       # <-- termination condition
            return decision.get("input", ""), state
        if action in specialists:
            out = specialists[action](decision.get("input", goal)).strip()
            state.append((action, out))
            if verbose:
                print(f"          {action} -> {out[:100]}...")
        else:
            state.append(("error", f"unknown action {action!r}"))

    return "[stopped: hit max_steps before 'done']", state   # <-- safety net

Run it. Watch the supervisor route researcher, writer, done on its own. That ordering was never written by us; it emerged from inspecting the state each turn.

In [ ]:
answer, state = supervise(
    "Produce a one-paragraph blurb about why sea otters matter to kelp forests.",
    max_steps=5,
)
print("\nFINAL ANSWER:\n", answer)

> Router vs supervisor: the router (08) is a single branch, one decision, then a worker runs and you're done. The supervisor is a loop, many decisions, re-evaluated against changing state. Use a router when one classification settles the work; use a supervisor when the next move genuinely depends on what just happened.

> The supervisor needs two exits: a clean one (`action == "done"`) and a hard one (`max_steps`). Without the hard exit, a confused supervisor that keeps calling the researcher will run until your token budget is gone. Every dynamic loop in this notebook has both.

## 4. Living with dynamic control flow

Dynamic flow is powerful, but it hands the model the steering wheel. Four risks recur, each with a simple, boring mitigation, and boring is exactly what you want here.

| Risk | What it looks like | Mitigation |
|---|---|---|
| Non-termination | a reflection or supervisor loop that never decides it's done | a hard cap (`max_rounds` / `max_steps`), always present, no exceptions |
| Cost blow-up | each loop resends a growing history; a few "extra" rounds quietly 5x your tokens | low caps, short `max_tokens`, only loop where quality justifies it; track `usage` |
| Error propagation | step 2 builds on a wrong step 1; one bad agent poisons the chain | validate each output (shape, length, gate) before passing it on; fall back or replan |
| Bad decisions | planner emits a nonsense step; supervisor picks an unknown action | validate the decision itself: check the JSON shape and that the action is in your registry, and have a fallback branch (you saw the `"unknown action"` and `[stopped]` branches above) |

> Rule of thumb: every loop gets a cap. Every inter-agent message gets validated before it's trusted. Every decision gets a fallback. These three habits are the difference between a demo and something you'd run unattended.

> Why you care: convergence is the goal. The system should measurably get closer to done each round. If it isn't (the critic keeps flipping, the supervisor keeps repeating itself), that's a signal to stop and rethink the criteria, not to raise the cap. A higher cap on a non-converging loop just costs more before failing.

## Recap

You can now build systems whose control flow is decided at runtime by the model, not fixed in advance:
- Planner to executor: an explicit, inspectable JSON plan, then step-by-step execution with results carried forward
- Reflection / self-critique: *generate, critique, revise* until a quality gate passes, with measurable improvement and a mandatory round cap
- Supervisor / manager: an agent that repeatedly chooses the next specialist from the current state, with both a `done` exit and a `max_steps` safety net
- The failure modes of dynamic flow (non-termination, cost blow-ups, error propagation, bad decisions) and the caps, validation, and fallbacks that contain them

### Try it yourself
1. Validate the plan. In `plan_task`, reject plans that aren't a list of 2 to 4 non-empty strings, and re-ask the planner once if validation fails.
2. Tighten the rubric. Add a 4th criterion to the reflection example (e.g. "must contain a verb") and watch how many rounds it now takes. Does it still converge within `max_rounds`?
3. Add a third specialist. Give the supervisor an `"editor"` that shortens text, and a goal that should make it choose researcher, writer, editor, done. Did it?
4. Count the cost. Make `reflect` and `supervise` return the number of LLM calls they made, and compare a 1-round vs 3-round reflection run.

### Next up
**10 · Capstone**: we assemble everything (a robust single agent, multiple specialists, static and dynamic orchestration) into one complete multi-agent system, then look at where to go from here: real frameworks (LangGraph, the OpenAI Agents SDK, CrewAI, AutoGen) and what to learn next.